# Worked example · Test the finished app like a stranger

Start the finished app from the notebook, then call it the way another program (or a visitor) would, using good input and bad input.

In [1]:
from pathlib import Path

import churn_model

if not Path("churn_model.joblib").exists():
    churn_model.main()  # trains in THIS environment and saves the model
else:
    print("churn_model.joblib already exists")

churn_model.joblib already exists


In [2]:
from gradio_client import Client

import app

app.demo.launch(prevent_thread_lock=True, quiet=True)
client = Client(app.demo.local_url, verbose=False)
print("running at", app.demo.local_url)

running at http://127.0.0.1:7860/


## A good customer, and a loyal one

In [3]:
def ask(*values):
    chances, action = client.predict(*values, api_name="/score")
    scores = {c["label"]: round(c["confidence"], 3) for c in chances["confidences"]}
    return scores, action


print(ask(3, 95.0, 4, "month-to-month", "cheque", 0.5))
print(ask(60, 40.0, 0, "two-year", "card", 0.5))

({'Will leave': 0.929, 'Will stay': 0.071}, 'Flag for a retention call')
({'Will stay': 0.959, 'Will leave': 0.041}, 'No action needed')


## The threshold changes the action, not the probability

In [4]:
print(ask(12, 70.0, 1, "month-to-month", "card", 0.2))
print(ask(12, 70.0, 1, "month-to-month", "card", 0.9))

({'Will leave': 0.672, 'Will stay': 0.328}, 'Flag for a retention call')


({'Will leave': 0.672, 'Will stay': 0.328}, 'No action needed')


## Bad input
A `ValueError` in `predict.py` becomes a `gr.Error` with a message a visitor can act on.

In [5]:
for values in [
    (-5, 95.0, 4, "month-to-month", "cheque", 0.5),
    (600, 95.0, 4, "month-to-month", "cheque", 0.5),
]:
    try:
        ask(*values)
    except Exception as error:
        print(type(error).__name__, "->", error)

AppError -> Tenure must be a number from 0 to 120 months.


AppError -> Tenure must be a number from 0 to 120 months.


## An empty monthly charge
The model can score it, so the app accepts it (and shows a warning in the browser).

In [6]:
print(ask(3, None, 4, "month-to-month", "cheque", 0.5))

({'Will leave': 0.898, 'Will stay': 0.102}, 'Flag for a retention call')


In [7]:
app.demo.close()

Closing server running on port: 7860


Then run the tests and the app from a terminal:

    python -m pytest -q
    python app.py